Phase 4 Don't Need the GPU , Implementation will be continued using the local CPU

In [2]:
# ==========================================
# 1. IMPORTS
# ==========================================

import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import kaggle

from pathlib import Path
from dotenv import load_dotenv

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ==========================================
# 2. PROJECT CONFIGURATION
# ==========================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ==========================================
# 3. ENVIRONMENT VARIABLES
# ==========================================

load_dotenv(PROJECT_ROOT / ".env")

HF_TOKEN = os.getenv("HF_TOKEN")
KAGGLE_API_TOKEN = os.getenv("KAGGLE_API_TOKEN")

# ==========================================
# 4. RANDOM SEED
# ==========================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Project initialized successfully!")

c:\Users\INDAML\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project initialized successfully!


Configure model and dataset

In [6]:
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

DATA_DIR = PROJECT_ROOT / "data" / "prepraed"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

Download Dataset from Kaggle

In [ ]:

# Download dataset from Kaggle

DATA_DIR = PROJECT_ROOT / "data" / "prepared"
DATA_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "dgowda11/sms-spam-cleaned"

!kaggle datasets download -d {KAGGLE_DATASET} -p "{DATA_DIR}" --unzip

print("Dataset downloaded to:", DATA_DIR)
print("Files:", list(DATA_DIR.iterdir()))


Load DataSet

In [8]:
train_df  = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Train DataFrame shape:", train_df.shape)
print("Test DataFrame shape:", test_df.shape)

train_df.head()

Train DataFrame shape: (4135, 2)
Test DataFrame shape: (1034, 2)


,label,message
0,ham,Cool. So how come you havent been wined and di...
1,ham,Huh so late... Fr dinner?
2,ham,my ex-wife was not able to have kids. Do you w...
3,ham,Hope ur head doesn't hurt 2 much ! Am ploughin...
4,ham,Ryder unsold.now gibbs.


Create the Prompt Formatter 

In [9]:
# Load the Tokenizer from Hugging Face

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("System prompt:", tokenizer.vocab_size)
print("EOS Token:", tokenizer.eos_token)

c:\Users\INDAML\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\INDAML\.cache\huggingface\hub\models--meta-llama--Llama-3.2-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer loaded: TokenizersBackend
System prompt: 128000
EOS Token: <|eot_id|>


In [14]:
# Instruction formatting configuration 

SYSTEM_PROMPT = (
    "You are an SMS Classification assistant."
    "Classify the message as HAM or SPAM."
    "Respond only with HAM or SPAM."
)

def format_sms_example(message, label):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Message: {message}\nLabel: {label}"
        },
        {
            "role": "assistant",
            "content": label.upper()
        }
    ]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenizer=False,
        add_generation_prompt=False
    )
    return formatted_text

                                                   

In [15]:

# Inspect one HAM and one SPAM sample

ham_sample = train_df[train_df["label"] == "ham"].iloc[0]
spam_sample = train_df[train_df["label"] == "spam"].iloc[0]

print("HAM SAMPLE:\n")
print(format_sms_example(ham_sample["message"], ham_sample["label"]))

print("\n" + "=" * 80 + "\n")

print("SPAM SAMPLE:\n")
print(format_sms_example(spam_sample["message"], spam_sample["label"]))


HAM SAMPLE:

{'input_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 868, 4723, 220, 2366, 21, 271, 2675, 527, 459, 31009, 55982, 18328, 19891, 1463, 279, 1984, 439, 473, 1428, 477, 9440, 1428, 8542, 3595, 1193, 449, 473, 1428, 477, 9440, 1428, 13, 128009, 128006, 882, 128007, 271, 2097, 25, 24882, 13, 2100, 1268, 2586, 499, 31081, 306, 1027, 289, 1619, 323, 294, 1619, 1603, 5380, 2535, 25, 13824, 128009, 128006, 78191, 128007, 271, 74593, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


SPAM SAMPLE:

{'input_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 868, 4723, 220, 2366, 21, 271, 2675, 527, 459, 31009, 55982